In [ ]:
from openpyxl import load_workbook
import pandas as pd
import unicodedata

In [ ]:
df = pd.read_excel('Libro1.xlsx')
df.columns = df.columns.str.strip()

wb = load_workbook('Libro1.xlsx')
ws = wb['BASE DE DATOS']

fecha_inicio = 'Fecha de Inicio del Contrato de Interventoría'
fecha_fin = 'Fecha de Finalización del Contrato de Interventoría'

headers = [
    cell.value.strip() if isinstance(cell.value, str) else cell.value
    for cell in ws[1]
]

NameError: name 'pd' is not defined

In [ ]:
col_inicio = headers.index(fecha_inicio) + 1
col_fin = headers.index(fecha_fin) + 1

def tiene_color(celda):
    fill = celda.fill

    if fill is None:
        return False

    if fill.fill_type is None:
        return False

    color = fill.fgColor.rgb

    if color in [None, '00000000', 'FFFFFFFF']:
        return False

    return True

estados = []

for row in range(2, ws.max_row + 1):
    celda_inicio = ws.cell(row=row, column=col_inicio)
    celda_fin = ws.cell(row=row, column=col_fin)

    if tiene_color(celda_inicio) or tiene_color(celda_fin):
        estados.append('FALSO')
    else:
        estados.append('ACTIVO')

df['Contrato Estado'] = estados

In [ ]:

df = df.dropna(
    subset=[
        'Nombres y Apellidos',
        'No. Cedula de ciudadanía',
        'No. Contrato de Interventoría'
    ],
    how='all'
)

In [ ]:
df['Nombres y Apellidos'] = (
    df['Nombres y Apellidos']
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)
df['Nombres y Apellidos'] = df['Nombres y Apellidos'].apply(
    lambda x: ''.join(
        c for c in unicodedata.normalize('NFD', x)
        if unicodedata.category(c) != 'Mn'
    ) if pd.notna(x) else x
)
df['Nombres y Apellidos'] = df['Nombres y Apellidos'].str.replace(r'[^A-ZÑ ]', '', regex=True)

In [ ]:
df['No. Cedula de ciudadanía'] = (
    df['No. Cedula de ciudadanía']
    .astype('string')
    .str.replace(r'\D', '', regex=True)
)
df['No. Cedula de ciudadanía'] = df['No. Cedula de ciudadanía'].replace('', pd.NA)

In [ ]:
col = 'No. Contrato de Interventoría'

df[col] = (
    df[col]
    .astype('string')
    .str.upper()
    .str.replace(r'-20\d{2}$', '', regex=True)
    .str.replace(r'\bNO\.\s*', '', regex=True)
    .str.replace('-', '', regex=False)
    .str.replace(r'\D', '', regex=True)
)

df[col] = df[col].fillna('').replace('', '0')
df.loc[df[col].str.len() < 3, col] = '0'
df.loc[df[col].str.fullmatch(r'0+', na=False), col] = '0'

In [ ]:
df['Año del Contrato de Interventoría'] = (
    df['Año del Contrato de Interventoría']
    .astype('string')
    .str.extract(r'(19\d{2}|20\d{2})')[0]
    .fillna('0')
)